# Weeks 7 and 8 - Visualisation

**Social and Cultural Analytics (7AAVBCS5)**

Use London's venue and bike-hire data to practise bar plots, histograms, line plots and scatter plots. For each chart, ask what it helps us see and what we can reasonably say from it.

Work through sections 1 to 5 in order. Sections 6 and 7 are optional. You can refer to the lecture and live coding for the plotting patterns. Most written answers need only one or two sentences, and you can finish the worksheet after the seminar.

Copy the download's data and weeks folders into your module folder, merging their contents. In Colab, upload this notebook and the three CSV files, then run setup.

## Setup and data

Run the two supplied cells below. They import pandas and matplotlib, then load the venue file as `venues`, the borough context file as `boroughs`, and the bike-hire file as `trips`. You do not need to write the loader.

The second cell also prepares the existing date column for plotting. These lines are supplied; you are not expected to write the date conversion yourself.

In [ ]:
import subprocess
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

DATA_REPO = "https://github.com/markjhill/7AAVBCS5-data.git"

def data_path(name: str) -> Path:
    """Find a teaching CSV locally, beside a Colab notebook, or in the public checkout."""
    here = Path.cwd()
    for candidate in (here, *here.parents):
        for local in (candidate / "data" / name,
                      candidate / name, candidate / Path(name).name):
            if local.is_file():
                return local
    downloaded = Path("7AAVBCS5-data")
    if not downloaded.is_dir():
        print("Downloading the module data once...")
        subprocess.run(["git", "clone", "--depth", "1", DATA_REPO,
                        str(downloaded)], check=True)
    target = downloaded / name
    if not target.is_file():
        raise FileNotFoundError(f"Upload {Path(name).name} beside the notebook, or check the data folder.")
    return target

In [ ]:
venues = pd.read_csv(data_path("london_cultural_venues_2023.csv"))
boroughs = pd.read_csv(data_path("london_cultural_infrastructure_2023_borough_context.csv"))
trips = pd.read_csv(data_path("tfl-bike-hire/bike_trips_16-31Jul2025_with_borough.csv"))

# SUPPLIED: prepare the existing calendar-date column for the line plot.
trips["date"] = pd.to_datetime(trips["date"])

print("Venue records:", len(venues))
print("Borough areas:", len(boroughs))
print("Recorded hires:", len(trips))
print("Hire dates:", trips["date"].min(), "to", trips["date"].max())

### The three files

The venue data comes from the GLA Cultural Infrastructure Map 2023 release, largely captured in 2022. It contains venue records, so the same place may appear in more than one register. The borough context file has one row for each of 32 boroughs plus the City of London.

The bike-hire extract contains records starting on 16-31 July 2025. One row is one hire, not one rider. Some durations are zero or extremely long. Keep these records for now; we will investigate them in section 2.

| Table | Columns used here |
|---|---|
| `venues` | `venue_type`, `borough_name` |
| `boroughs` | `ptal_score_2014`, `venues_per_10k_residents`, `venues`, `population_2015` |
| `trips` | `date`, `hour`, `weekday`, `duration_min`, `bike_model`, `start_time` |

Duration is in minutes. `hour` contains whole numbers from 0 to 23. The venue rate uses the 2023 release and 2015 resident population; PTAL is from 2014. These are not measurements from the same year.

Sources: the module's prepared GLA venue and borough-context extracts, and the lecturer-supplied TfL extract.

## 1. Bar plots

We start with the venue data. The question is which categories appear most often in the records.

**Worked example.** Run the next cell and read it closely. It counts venue records by `venue_type`, keeps the eight most frequent as `top_types`, and draws them as horizontal bars with the largest at the top. You reuse this exact pattern in Task 1.1.

In [ ]:
# WORKED EXAMPLE. You adapt this pattern in Task 1.1.
type_counts = venues["venue_type"].value_counts()
top_types = type_counts.head(8)
print(top_types)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top_types.index, top_types.values, color="#0072B2")
ax.invert_yaxis()
ax.set_xlabel("Venue records")
ax.set_title("Eight most frequent venue types - 2023 release")
fig.tight_layout()

**Task 1.1.** Ask the same question of a different column: which eight **boroughs** hold the most venue records? Build `borough_counts` from `borough_name`, keep the eight largest as `top_boroughs`, print them, and draw the same horizontal bar chart with a title and axis label that name boroughs.

You change the column and the labels; the plotting lines are otherwise identical to the worked example. If you get a `KeyError`, check the name against `venues.columns`.

In [ ]:
# Write your code here

**Task 1.2.** Which borough has the most records, and how many? Compare its lead over the second-placed borough with how far `Archives` leads the second-placed venue type, first as a difference in records and then as a ratio. Do the two comparisons point the same way? Then, in one sentence using what both charts actually count, explain why neither shows the number of distinct cultural places.

*Write your answers here.*

## 2. Histograms

Now switch from counting categories to seeing how a numerical measurement is distributed.

**Worked example.** Run the next cell. It stores `trips["duration_min"]` as `durations`, summarises it, draws a 30-bin histogram of the full range, then keeps the first-hour window as `within_hour` and draws a 10-bin histogram of it. You reuse this pattern on a different measurement in Task 2.1.

In [ ]:
# WORKED EXAMPLE. You adapt this pattern in Task 2.1.
durations = trips["duration_min"]
print("Missing durations:", durations.isna().sum())
print(durations.describe().round(2))

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(durations, bins=30, color="#0072B2", edgecolor="white")
ax.set_xlabel("Recorded hire duration (minutes)")
ax.set_ylabel("Hires")
ax.set_title("Full recorded duration range - 443,539 hires")
fig.tight_layout()

within_hour = durations[durations <= 60]
print("Within first hour:", len(within_hour), "of", len(durations),
      "- outside:", len(durations) - len(within_hour))
print("Recorded zeros kept:", (within_hour == 0).sum())
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(within_hour, bins=10, color="#0072B2", edgecolor="white")
ax.set_xlabel("Recorded hire duration (minutes; 0–60 window)")
ax.set_ylabel("Hires")
ax.set_title("First-hour view - 432,667 hires")
fig.tight_layout()

**Task 2.1.** Use the same `.describe()` and `ax.hist(...)` pattern on a different measurement: the borough venue rate `venues_per_10k_residents` in `boroughs`. Store it as `rate`, print `.describe()`, and draw a 10-bin histogram. Label the measurement axis in venues per 10,000 residents (2015 population) and the count axis as borough areas.

This is a different table with far fewer rows, so write your own title and labels rather than reusing the bike-hire ones. If any value is missing, `.dropna()` before plotting, as in the lecture.

In [ ]:
# Write your code here

**Task 2.2.** Compare the two histograms. The durations have 443,539 rows and the rates only 33. Describe the shape of the rate distribution and name the area at the far right. What does the small number of rows make harder to judge than it was for the durations?

*Write your answers here.*

## 3. Line plots

Now follow one measurement over time.

**Worked example.** Run the next cell. It groups the hires by `date`, counts them with `.size()` as `daily`, and draws one point per calendar day. You reuse this pattern for a different daily summary in Task 3.1.

In [ ]:
# WORKED EXAMPLE. You adapt this pattern in Task 3.1.
daily = trips.groupby("date").size()
print(daily)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(daily.index, daily.values, marker="o", color="#0072B2")
ax.set_xlabel("Hire start date")
ax.set_ylabel("Hires started per day")
ax.set_title("Daily recorded hires - 16–31 July 2025")
fig.tight_layout()

**Task 3.1.** Keep the daily line, but change what it measures. Instead of counting hires, show the **mean hire duration** per day: group `trips` by `date`, take the mean of `duration_min`, store it as `daily_mean`, and plot it with `marker="o"`. Label the vertical axis in minutes.

You are changing `.size()` to a mean of one column: `trips.groupby("date")["duration_min"].mean()`. If you forget to pick the column, you summarise every column at once, so select `duration_min`.

In [ ]:
# Write your code here

**Task 3.2.** A mean can be pulled up by a few very long hires. Calculate the daily **median** duration as well, with the same pattern but `.median()` in place of `.mean()`, and store it as `daily_median`. Print the mean and the median for the busiest day (16 July) and the quietest day (19 July) side by side.

Use `.loc["2025-07-16"]` and `.loc["2025-07-19"]` on each series to read a single day.

In [ ]:
# Write your code here

**Task 3.3.** The mean duration is higher on the quiet day than on the busy day. Does the median agree? Using both numbers, say whether the busy day really has shorter *typical* hires, and explain what the gap between the mean and the median on the quiet day suggests. Then state why two days cannot establish a general pattern.

*Write your answers here.*

## 4. Scatter plots and correlation

Return to the borough data to ask whether two measurements move together.

**Worked example.** Run the next cell. It keeps the complete `ptal_score_2014` and `venues_per_10k_residents` pairs as `pair`, draws a scatter, and reports Pearson and Spearman. You reuse this pattern on a different pair of columns in Task 4.1.

In [ ]:
# WORKED EXAMPLE. You adapt this pattern in Task 4.1.
pair = boroughs[["ptal_score_2014", "venues_per_10k_residents"]].dropna()
print("Complete pairs:", len(pair))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.scatter(pair["ptal_score_2014"], pair["venues_per_10k_residents"],
           s=35, color="#0072B2")
ax.set_xlabel("Public transport accessibility (PTAL, 2014)")
ax.set_ylabel("Venue records per 10,000 residents (2015)")
ax.set_title("Accessibility and venue rates - 33 areas")
fig.tight_layout()

print("Pearson:", round(pair["ptal_score_2014"].corr(pair["venues_per_10k_residents"]), 3))
print("Spearman:", round(pair["ptal_score_2014"].corr(
    pair["venues_per_10k_residents"], method="spearman"), 3))

**Task 4.1.** Does the comparison change when we count venue records rather than express them per resident? Keep the complete `ptal_score_2014` and `venues` pairs from `boroughs` as `count_pair`. Print n, draw the scatter, and calculate Pearson and Spearman rounded to three places. Describe the direction and shape before reading the coefficients.

Adapt the worked example: use `count_pair` throughout and replace `venues_per_10k_residents` with `venues`. Label the vertical axis as venue records from the 2023 release. Keep PTAL on the horizontal axis.

In [ ]:
# Write your code here

**Task 4.2.** Report n and both coefficients for your count scatter. Compare it with the worked example's rate scatter. Does the direction change? Explain what Pearson and Spearman each summarise, referring to the patterns you can see. Why are counts and resident rates different questions, even if their coefficients are similar? Give one reason neither plot establishes that transport creates venues.

*Write your answers here.*

## 5. Save a figure and write a caption

**Task 5.1.** Save your count scatter from Task 4.1 as `combined_workshop_scatter.png`. Open the exported file. Are the labels readable and complete at the size you would use in a report?

Use `fig.savefig("combined_workshop_scatter.png", dpi=200)`. If you have drawn another figure since Task 4.1, rerun that cell first so `fig` refers to the count scatter.

In [ ]:
# Write your code here

**Task 5.2.** Write a caption for your count scatter naming the observation unit, n, both measures and their years, the two coefficients you calculated, and one limit on interpretation. Then give one sentence for each of the four plot types you used, saying why it suited its question.

*Write your answers here.*

## 6. Optional: take the comparisons further

The core worksheet is finished. Try these if you have time, or come back to them after the seminar. They use the same lecture methods and data.

**Task 6.1.** Compare Lambeth, Southwark and Lewisham using grouped counts and proportional stacked bars. Make a full borough-by-type crosstab, select `Libraries`, `Theatre` and `Cinemas`, and add `Other venue types` as the full borough total minus the selected counts. Keep the colours and category order the same in both charts, and print the borough totals.

Which chart makes it easier to compare library counts, and which shows the mix of entries? Compare Southwark and Lewisham's library counts and shares. Why does keeping Other matter?

Use `pd.crosstab()`, `.loc[rows, columns].copy()` and `.sum(axis=1)`. For the plots, use `table.plot.bar(ax=ax, ...)`; add `stacked=True` for stacked bars. Calculate percentages with `.div(table.sum(axis=1), axis=0) * 100`.

In [ ]:
# Write your code here

*Write your answers here.*

**Task 6.2.** Draw ten-bin and thirty-bin histograms of the same `within_hour` durations. Keep the horizontal range at 0-60 minutes. Add mean and median lines with different line styles and a legend, calculating both from `within_hour`. Print the mean and median for the full data as well.

What changes when the bins change, and what changes when the included hires change? Why is the tallest bin not necessarily the numerical mode?

Use `ax.set_xlim(0, 60)`, `.mean()`, `.median()`, `ax.axvline(..., label=...)` and `ax.legend()`. The lecture used `linestyle="--"` for the mean and `linestyle=":"` for the median.

In [ ]:
# Write your code here

*Write your answers here.*

**Task 6.3.** Compare the hourly pattern on weekdays and weekends. Select Saturday and Sunday hires as `weekend_trips`, and Monday to Friday hires as `weekday_trips`, using `.isin()` as in the lecture. Group each subset by `hour`, count the hires, and divide by 4 weekend dates or 12 weekday dates. Draw both series on one set of axes with a legend.

Why would comparing the raw totals be uneven? Describe the peaks in each line. What might they suggest about commuting, and what can they not establish?

Use `subset.groupby("hour").size()` and two calls to `ax.plot(..., label=...)`, followed by `ax.legend()`. These denominators describe this extract, not every dataset.

In [ ]:
# Write your code here

*Write your answers here.*

**Task 6.4.** A third measurement can be shown by marker size. Select `ptal_score_2014`, `venues_per_10k_residents` and `population_2015` from `boroughs`, drop incomplete rows as `sized`, and redraw the worked example's rate scatter with each marker scaled by population.

Use `s=sized["population_2015"] / 1500` in `ax.scatter(...)`, as in the lecture. State this scale beside your plot: marker area represents resident population divided by 1500.

What does adding population show that the plain scatter did not? Does a larger marker mean a higher venue rate? Identify the highest-rate area and compare its marker size with the others.

In [ ]:
# Write your code here

*Write your answers here.*

**Supplied prep for Task 6.5.** Run the next cell. It prepares fractional start hours, keeps hires lasting at most 60 minutes, and supplies the bike-model colours. You are not expected to write the time conversion or colour preparation.

In [ ]:
# SUPPLIED: prepare start hours and model colours for Task 6.5.
started = pd.to_datetime(trips["start_time"])
trips["start_hour"] = started.dt.hour + started.dt.minute / 60
within_hour_trips = trips[trips["duration_min"] <= 60].copy()
within_hour_trips["model_colour"] = within_hour_trips["bike_model"].map(
    {"CLASSIC": "steelblue", "PBSC_EBIKE": "orange"})
print("Hires to plot:", len(within_hour_trips))
print("Blue: CLASSIC; orange: PBSC_EBIKE")

**Task 6.5.** Draw `start_hour` against `duration_min` for `within_hour_trips`, first with `s=6`, then on a fresh figure with `s=2, alpha=0.02`. Keep the same observations in both plots. What becomes visible when the markers are smaller and more transparent?

Draw the transparent version once more, adding `c=within_hour_trips["model_colour"]`. The supplied column uses blue for CLASSIC and orange for PBSC_EBIKE. State these colours beside the plot. Does colouring by model suggest separate patterns or substantial overlap? Explain what this plot cannot tell you about speed or journey purpose.

Use the lecture's `ax.scatter(...)` pattern. Label the horizontal axis in hours since midnight and the vertical axis in minutes. The colour column is supplied, so you do not need to construct a legend or prepare the colours yourself.

In [ ]:
# Write your code here

*Write your answers here.*

## 7. Work with your own data

Optional. Use a dataset you are considering for your project. These tasks bring together the EDA methods from the lectures. There is no single right answer because the dataset and question are yours.

Start with a question you would like to investigate. Use the plots that suit your variables. If a task does not fit your data, explain why and what extra measurement you would need. An identifier is not a numerical measurement just because it contains numbers.

**Task 7.1.** Load your dataset as `data`. Inspect its first rows, shape and column names. In one or two sentences, say what one row represents, where the data came from, and what question you would like to investigate.

Use `pd.read_csv()`, `.head()`, `.shape` and `.columns`, as in Week 3. Use your own file path rather than the module's `data_path()` helper.

In [ ]:
# Write your code here

*Write your answers here.*

**Task 7.2.** Check missing values in the columns relevant to your question. Choose one categorical variable and one numerical measurement, then inspect their frequency table and numerical summary. What needs checking before you use them?

Use `.isna().sum()`, `.value_counts()` and `.describe()`. Look at possible labels, units, unusual values and the number of observations available. A zero and a missing value are not necessarily the same thing.

In [ ]:
# Write your code here

*Write your answers here.*

**Task 7.3.** Make a bar plot of your categorical variable. Decide whether to order the categories by frequency or keep a meaningful order, and choose an orientation that makes the labels readable. State if you show only some categories.

Which category is most frequent, with how many records? What exactly are those records counting? If you report a percentage, give its denominator.

Adapt the frequency-table and plotting patterns from section 1.

In [ ]:
# Write your code here

*Write your answers here.*

**Task 7.4.** Draw a histogram of your numerical measurement. Try two bin settings on the same observations, then compare the mean and median. Describe the shape and identify any values that need investigating.

What remains visible when the bins change? If you use a subset or viewing window, state the rule and how many observations it leaves out. Keep the original data available.

Adapt section 2 and Task 6.2. Label the measurement's units and the number of observations, rather than copying the bike-hire labels.

In [ ]:
# Write your code here

*Write your answers here.*

**Task 7.5.** If your data has a suitable time or other ordered numerical variable, make a line plot relevant to your question. Decide whether each point should show a count, a total or a typical value, and summarise the data accordingly.

Check the order and spacing before joining points. What does the interval between them represent, and what change can you see? If you compare groups observed for different amounts of time, explain how you made the comparison fair. If there is no suitable variable, explain why a line would be misleading.

Use `.groupby()` with `.size()`, `.sum()`, `.mean()` or `.median()`, then `ax.plot()`. If your dates need preparation you have not learned, ask for help rather than assuming text labels are in date order.

In [ ]:
# Write your code here

*Write your answers here.*

**Task 7.6.** Choose two numerical measurements that might be related. Keep complete pairs, report n and draw a scatter plot. Describe its direction, shape and unusual observations before calculating Pearson and Spearman.

Which coefficient would you emphasise for your question, and why? Refer to the plot and the difference between values and ranks, rather than choosing the larger number. State one thing the association cannot establish. If you do not have a meaningful pair, say what measurements would be needed.

Use the selection, `.dropna()`, plotting and `.corr()` patterns from section 4.

In [ ]:
# Write your code here

*Write your answers here.*

**Task 7.7.** Choose a category and a numerical measurement to compare across groups. Calculate each group's count, mean and median, then choose a chart that helps answer your question.

Do the groups have similar numbers of observations? Do the mean and median suggest the same comparison? Explain why your chart helps us read the result, and name one thing the group summaries might hide.

Use `.groupby()` and `.agg(["count", "mean", "median"])`. You can plot summary columns as grouped bars using the lecture pattern and Task 6.1. Keep the group counts visible beside the chart.

In [ ]:
# Write your code here

*Write your answers here.*

**Task 7.8.** Choose two useful figures from your exploration. Save them, open the exported files and check that the labels and units are readable. Write a short data note of about 150 words using the actual results.

State what the observations represent, report two findings with their values and denominators where relevant, and explain one limitation or unresolved data issue. End with one question you would investigate next. Explain how the figures helped you reach the findings rather than just describing the steps in your code.

Rerun the plotting cell for each chosen figure before saving its `fig`, using the pattern from section 5.

In [ ]:
# Write your code here

*Write your answers here.*

## Before you finish

**Restart and Run All** to check that your completed notebook runs from top to bottom. Save the notebook and any figures you exported.